# FCN 논문 구현 및 Segmentation 실습

# 준비

In [ ]:
!unzip flood.zip

In [ ]:
import os, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from glob import glob
from PIL import Image
from tqdm.notebook import tqdm
import torch, torchvision
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms

In [ ]:
image_list = glob('./Image/*')
mask_list = glob('./Mask/*')
len(image_list), len(mask_list)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for i in range(4):
    img_path = image_list[i]
    mask_path = image_list[i].replace('Image', 'Mask').replace('jpg', 'png')

    axes[2 * i].imshow(Image.open(img_path))
    axes[2 * i].set_title(f'Image {i+1}')
    axes[2 * i].axis('off')

    axes[2 * i + 1].imshow(Image.open(mask_path))
    axes[2 * i + 1].set_title(f'Mask {i+1}')
    axes[2 * i + 1].axis('off')

In [ ]:
unique, counts = np.unique(mask_list, return_counts=True)
for mask_path in mask_list:
    mask = Image.open(mask_path)
    mask = np.array(mask)
    print(mask)
    print(np.unique(mask))
    break

In [ ]:
exclude = []
for img_path in image_list:
    img = Image.open(img_path)
    img = np.array(img)
    if img.ndim != 3 or img.shape[2] != 3:
        print(img.shape)
        print(img.ndim)
        print(img_path)
        exclude.append(img_path)

### 차원 불일치 데이터 제거

In [ ]:
for ex in exclude:
    image_list.remove(ex)
    mask_list.remove(ex.replace('Image', 'Mask').replace('jpg', 'png'))

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, image_list, mask_list, transform):
        self.image_list = image_list
        self.mask_list = mask_list
        self.transform = transform
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, idx):
        image_path = self.image_list[idx]
        mask_path = image_path.replace('Image', 'Mask').replace('jpg', 'png')
        image = Image.open(image_path)
        mask = Image.open(mask_path)
        image = self.transform(image)
        mask = self.transform(mask)
        mask = (mask > 0.5).float()
        return image, mask

In [ ]:
transform = transforms.Compose([
    # class index는 정수 사용(실수 금지)
    # 나누거나 평균/표준편차로 정규화하면 학습 불안정 or 완전 실패
    transforms.ToTensor(), # 255로 나누어 [0, 1]범위로 정규화
transforms.Resize((224, 224)), ])
flood_dataset = FloodDataset(image_list, mask_list,
transform=transform)
print(len(flood_dataset))
train_size = int(0.8 * len(flood_dataset))


In [ ]:
flood_dataset = FloodDataset(image_list, mask_list,
transform=transform)
print(len(flood_dataset))
train_size = int(0.8 * len(flood_dataset))
val_size = len(flood_dataset) - train_size
train_ds, val_ds = random_split(flood_dataset, [train_size,
val_size])
print(f"Train dataset size: {len(train_ds)} | "
f"Validation dataset size : {len(val_ds)}")
train_dl = DataLoader(train_ds, batch_size = 16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size = 16, shuffle=False)
imgs, masks = next(iter(train_dl))
print(imgs.shape, masks.shape)


## 학습 및 추론


### 모델 구현
아래 그림과 조건에 맞게 모델을 구현해봅니다.
![](https://velog.velcdn.com/images%2Fleejaejun%2Fpost%2Fc3b69a0d-2329-4903-9baa-de2949af87fb%2Fimage.png)

- VGG16 Backbone을 활용
    - VGG16 Backbone은 총 5개의 Conv block으로 구성
    - 이미지를 Backbone에 통과
    - 추가 Conv layer 2개를 통과
    - 마지막 Feature map Channel size = class num과 같아야 함
- Skip connection
    - 3, 4번 block을 통과한 Feature map을 재사용
    - Upsampling Feature map과 더하기 위하여 Channel size 조정 필요
    - 결합 방식: tensor_1 + tensor_2(더하기 연산)
- Upsampling
    - 두 개의 Upsampling 레이어를 통과
    - 총 세 번 Upsampling(2배 증가 2번, 8배 증가 1번)
    - 첫 번째 2배 증가 후 4번 Block과 결합
    - 두 번째 2배 증가 후 3번 Block과 결합

In [ ]:
class FCN8s(nn.Module):
  def __init__(self, n_classes):
      super().__init__()
      vgg = models.vgg16(pretrained=True)
      features = list(vgg.features.children())
      # VGG16의 각 블록을 PyTorch Sequential로 구성
      self.block3 = nn.Sequential(*features[:17]) # conv1~conv3
      self.block4 = nn.Sequential(*features[17:24]) # conv4
      self.block5 = nn.Sequential(*features[24:]) # conv5 ~
      # 추가 Conv 레이어(우리 응용에 필요한 레이어)
      self.conv6 = nn.Conv2d(512, 4096, kernel_size=7, padding=3)
      self.conv7 = nn.Conv2d(4096, 4096, kernel_size=1)
      # FCN에서 사용할 1x1 Conv
      self.conv1x1_pool3 = nn.Conv2d(256, n_classes, kernel_size=1)
      self.conv1x1_pool4 = nn.Conv2d(512, n_classes, kernel_size=1)
      self.conv1x1_output = nn.Conv2d(4096, n_classes, kernel_size=1)
      # Transposed Convolutions for upsampling
      self.upconv2 = nn.ConvTranspose2d(n_classes, n_classes,
kernel_size=4, stride=2, padding=1)
      self.upconv8 = nn.ConvTranspose2d(n_classes, n_classes,
kernel_size=8, stride=8, padding=0)
  def forward(self, x):
    # VGG16의 Backbone 통과
      pool3 = self.block3(x)
      pool4 = self.block4(pool3)
      pool5 = self.block5(pool4)
      pool5 = F.relu(self.conv6(pool5))
      pool5 = F.relu(self.conv7(pool5))
      # FCN 의 Head (Decoder 부분)
      score = self.conv1x1_output(pool5) # (1, H/32, W/32) 7,7
      score = self.upconv2(score) + self.conv1x1_pool4(pool4)  # 14 x 14
      score = self.upconv2(score) + self.conv1x1_pool3(pool3)  # 28 x 28
      output = self.upconv8(score)  # 224 x 224
      return output

In [ ]:
# 모델 인스턴스 생성
model = FCN8s(n_classes=1).cuda()
input_image = torch.randn(1, 3, 224, 224).cuda()
output = model(input_image)
print(output.shape)

## 7. 학습 루프 설정

In [ ]:
from torch import optim
criterion = nn.BCEWithLogitsLoss()
optim = optim.Adam(model.parameters(), lr=5e-4) # lr = 1e-4, 1e-5, 5e-5
epochs =20

## 8. 평가 지표

In [ ]:

def IoU(output, mask):
  """
  1. pred를 threshold 기준 Binary 분류
  2. pred와 gt사이 교집합 영역(픽셀) 계산
  3. pred와 gt사이 합집합 영역(픽셀) 계산
  4. 교집합 영역 / 합집합 영역 계산(zero division 방지)
  """
  output = (output > 0.5).float() # 모델 예측값이 0 or 1
  intersection = torch.sum(output * mask) # 교집합
  union = torch.sum(output) + torch.sum(mask) - intersection # 합집합
  return intersection / (union + 1e-9)
def PA(output, mask):
  """
  1. pred를 threshold 기준 Binary 분류
  2. pred와 gt사이 일치 여부 계산
  3. 전체 픽셀 중 일치 픽셀 수 반환
  """
  output = (output > 0.5).float() # 모델 예측값이 0 or 1
  correct = torch.sum(output == mask) # True 숫자
  total = torch.numel(mask) # 요소 개수
  return correct / (total+ 1e-9)

## 14. 모델 가중치 불러오기(있을때만)

In [ ]:
# 주의하실 점: 저장 당시 GPU, 불러올때는 CPU라면 map_location='cpu'추가
import torch
#저장된 체크포인트가 현재 폴더 밑에 model.pth일 때
checkpoint = torch.load('model.pth')
#CPU라면, checkpoint = torch.load('model.pth', map_location='cpu')
# 모델과 옵티마이저 상태 복원
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# 옵티마이저는 없을 수도 있으니 조건 처리
if 'optimizer_state_dict' in checkpoint:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
else:
    print("Optimizer state not found, starting fresh.")
# epoch 정보도 함께 불러올 수 있음
START_epoch = checkpoint['epoch']

## 9. 학습 루프 함수

In [ ]:
def train_and_validate(model, train_loader, val_loader, optim, criterion, epochs):
  train_losses = []
  train_ious = []
  train_pas = []
  val_losses = []
  val_ious = []
  val_pas = []
  for epoch in range(START_epoch, START_epoch + epochs): # Epoch loop
    train_loss, train_iou, train_pa = 0, 0, 0
    model.train()
    for images, masks in tqdm(train_loader): # Step loop
      images, masks = images.cuda(), masks.cuda()
      optim.zero_grad() # 이전 루프 미분값 청소
      outputs = model(images)
      loss = criterion(outputs, masks)
      loss.backward()# 현재 루프 미분값 계산
      optim.step()# 현재 루프 미분값 가중치에 업데이트
      # 로그 기록
      train_loss += loss.item()
      train_iou += IoU(outputs, masks).item()
      train_pa += PA(outputs, masks).item()
    print(f"Epoch {epoch + 1} \t Training Loss: {train_loss / len(train_loader):.2f} \t Training IoU: {train_iou / len(train_loader):.2f} \t Training PA: {train_pa / len(train_loader):.2f}")
    train_losses.append(train_loss / len(train_loader))
    train_ious.append(train_iou / len(train_loader))
    train_pas.append(train_pa / len(train_loader))
    val_loss, val_iou, val_pa = 0, 0, 0
    model.eval()
    with torch.no_grad():
      for images, masks in tqdm(val_loader):
        images, masks = images.cuda(), masks.cuda()
        outputs = model(images) # 순전파(feed forward)
        loss = criterion(outputs, masks)
        val_loss += loss.item()
        val_iou += IoU(outputs, masks).item()
        val_pa += PA(outputs, masks).item()
      print(f"Epoch {epoch + 1} \t Validation Loss: {val_loss / len(val_loader):.2f} \t Validation IoU: {val_iou / len(val_loader):.2f} \t Validation PA: {val_pa / len(val_loader):.2f}")
      val_losses.append(val_loss / len(val_loader))
      val_ious.append(val_iou / len(val_loader))
      val_pas.append(val_pa / len(val_loader))
  return train_losses, train_ious, train_pas, val_losses, val_ious, val_pas

## 10. 학습

In [ ]:
train_losses, train_ious, train_pas, val_losses, val_ious, val_pas = train_and_validate(model, train_dl, val_dl, optim, criterion, epochs)

## 11. 로그 시각화

In [ ]:
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.plot(train_ious, label='Training IoU')
plt.plot(val_ious, label='Validation IoU')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.legend()
plt.plot(train_pas, label='Training PA')
plt.plot(val_pas, label='Validation PA')
plt.xlabel('Epoch')
plt.ylabel('PA')
plt.show()

## 12. 영상, gt, pred 결과 비교

In [ ]:
def plot_batch(model, data_loader):
  model.eval()
  with torch.no_grad():
    for images, masks in data_loader:
      # images, masks = images.cuda(), masks.cuda()
      outputs = model(images)
      outputs = (outputs > 0.5).float()
      img = torchvision.utils.make_grid(images).cpu()
      mask = torchvision.utils.make_grid(masks).cpu()
      output = torchvision.utils.make_grid(outputs).cpu()
      img = img.permute(1, 2, 0) # .permute(1, 2, 0): 채널 차원을 맨 뒤로 이동
      mask = mask.permute(1, 2, 0)
      output = output.permute(1, 2, 0)
      fig, axes = plt.subplots(1, 3, figsize=(15, 5))
      axes[0].imshow(img)
      axes[0].set_title('Image')
      axes[1].imshow(mask, cmap='gray')
      axes[1].set_title('Ground Truth')
      axes[2].imshow(output, cmap='gray')
      axes[2].set_title('Predicted Mask')
      plt.show()
      break

## 13. 모델 가중치 저장

In [ ]:
torch.save({'epoch': 20, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()}, 'model.pth')